# 07 — Build Enhanced xG Datasets

Assemble final skill-aware, form-aware, and combined modeling datasets from the baseline,
skill, and form feature pipelines. Notebook 07 standardises the final model inputs so
notebook 08 can compare models fairly.

**Why this notebook is necessary:**
The pipeline outputs follow a cumulative layering pattern:
- Baseline (36 cols): baseline features + identity columns + `xg_pred`
- Skill (41 cols): baseline + 5 skill columns
- Form (49 cols): baseline + 5 skill + 8 form columns

The form parquet already contains skill features. Using it directly as the
"form-aware" dataset would give that model skill features — making a fair
comparison of skill vs form impossible. Notebook 07 reconstructs all three
datasets explicitly from baseline + incremental feature blocks:

| Assembled output | Formula | Cols |
|-----------------|---------|------|
| skill-aware | baseline + SKILL_COLS | 36 + 5 = 41 |
| form-aware | baseline + FORM_COLS only | 36 + 8 = 44 (input form parquet is 49 — skill excluded) |
| combined | baseline + SKILL_COLS + FORM_COLS | 36 + 5 + 8 = 49 |

No new features are engineered. No models are trained. This is assembly only.

In [1]:
import sys
import os
import pandas as pd
from IPython.display import display

sys.path.append(os.path.abspath(".."))

from src.config import DATA_DIR, SKILL_COLS, FORM_COLS
from src.datasets import (
    build_skill_dataset, build_form_dataset, build_combined_dataset,
    save_dataset_bundle,
)

ID_COLS = ["league", "matchId", "playerId", "teamId", "eventSec", "matchPeriod"]

## Step 0: Path validation

In [2]:
BASE_TRAIN  = DATA_DIR / "wyscout_train_xg_baseline_predictions.parquet"
BASE_TEST   = DATA_DIR / "wyscout_test_xg_baseline_predictions.parquet"
SKILL_TRAIN = DATA_DIR / "wyscout_train_xg_skill.parquet"
SKILL_TEST  = DATA_DIR / "wyscout_test_xg_skill.parquet"
FORM_TRAIN  = DATA_DIR / "wyscout_train_xg_form.parquet"
FORM_TEST   = DATA_DIR / "wyscout_test_xg_form.parquet"

for path in [BASE_TRAIN, BASE_TEST, SKILL_TRAIN, SKILL_TEST, FORM_TRAIN, FORM_TEST]:
    assert path.exists(), f"Missing input file: {path}"
print("Paths OK")

Paths OK


## Step 1: Load all six inputs

In [3]:
base_train  = pd.read_parquet(BASE_TRAIN)
base_test   = pd.read_parquet(BASE_TEST)
skill_train = pd.read_parquet(SKILL_TRAIN)
skill_test  = pd.read_parquet(SKILL_TEST)
form_train  = pd.read_parquet(FORM_TRAIN)
form_test   = pd.read_parquet(FORM_TEST)

# Row count invariants stored immediately — referenced throughout
n_base_train = len(base_train)
n_base_test  = len(base_test)

# BASE_COLS: baseline columns to carry into every output (excludes dataset_split if present)
# n_base_cols is derived from BASE_COLS so it is future-proof against dataset_split appearing
assert "dataset_split" not in base_train.columns, "base_train has dataset_split before any processing"
assert "dataset_split" not in base_test.columns,  "base_test has dataset_split before any processing"
BASE_COLS   = [c for c in base_train.columns if c != "dataset_split"]
n_base_cols = len(BASE_COLS)   # column count anchor for Steps 3–5 and reload verification

print(f"base_train:  {base_train.shape}")
print(f"base_test:   {base_test.shape}")
print(f"skill_train: {skill_train.shape}")
print(f"skill_test:  {skill_test.shape}")
print(f"form_train:  {form_train.shape}")
print(f"form_test:   {form_test.shape}")
print(f"n_base_cols (column anchor): {n_base_cols}")

# Schema symmetry: each split pair must have the same columns
assert base_train.columns.tolist()  == base_test.columns.tolist(),  "base_train/test column mismatch"
assert skill_train.columns.tolist() == skill_test.columns.tolist(), "skill_train/test column mismatch"
assert form_train.columns.tolist()  == form_test.columns.tolist(),  "form_train/test column mismatch"
print("Schema symmetry OK")

base_train:  (34159, 36)
base_test:   (8881, 36)
skill_train: (34159, 41)
skill_test:  (8881, 41)
form_train:  (34159, 49)
form_test:   (8881, 49)
n_base_cols (column anchor): 36
Schema symmetry OK


## Step 2: Validate source integrity

In [4]:
BASE_REQUIRED  = set(ID_COLS + ["is_goal", "xg_pred"])
SKILL_REQUIRED = set(ID_COLS + SKILL_COLS)
FORM_REQUIRED  = set(ID_COLS + FORM_COLS)

for name, df, required in [
    ("base_train",  base_train,  BASE_REQUIRED),
    ("base_test",   base_test,   BASE_REQUIRED),
    ("skill_train", skill_train, SKILL_REQUIRED),
    ("skill_test",  skill_test,  SKILL_REQUIRED),
    ("form_train",  form_train,  FORM_REQUIRED),
    ("form_test",   form_test,   FORM_REQUIRED),
]:
    missing = required - set(df.columns)
    assert not missing, f"Missing columns in {name}: {missing}"

# Non-empty checks
for name, df in [
    ("base_train", base_train), ("base_test", base_test),
    ("skill_train", skill_train), ("skill_test", skill_test),
    ("form_train", form_train), ("form_test", form_test),
]:
    assert len(df) > 0, f"{name} is empty"

# League checks
for name, df in [("base_train", base_train), ("skill_train", skill_train), ("form_train", form_train)]:
    assert set(df["league"].unique()) == {"France","Germany","Italy","Spain"}, \
        f"Wrong train leagues in {name}"
for name, df in [("base_test", base_test), ("skill_test", skill_test), ("form_test", form_test)]:
    assert set(df["league"].unique()) == {"England"}, f"Wrong test league in {name}"

# Duplicate ID checks
for name, df in [
    ("base_train", base_train), ("base_test", base_test),
    ("skill_train", skill_train), ("skill_test", skill_test),
    ("form_train", form_train), ("form_test", form_test),
]:
    assert not df[ID_COLS].duplicated().any(), f"Duplicate IDs in {name}"

print("Source integrity OK")

Source integrity OK


## Step 3: Validate row-level alignment across sources

Before merging, verify that baseline, skill, and form inputs refer to the exact same
shot rows within each split. If this fails, stop rather than continuing with misaligned data.

In [5]:
def id_set(df):
    return set(map(tuple, df[ID_COLS].to_numpy()))

assert id_set(base_train) == id_set(skill_train) == id_set(form_train), \
    "Train shot identities do not align across inputs"
assert id_set(base_test)  == id_set(skill_test)  == id_set(form_test), \
    "Test shot identities do not align across inputs"

# Target consistency check — sort by ID_COLS to be order-independent
def check_target_match(df_a, df_b, col="is_goal"):
    a = df_a.set_index(ID_COLS)[col].sort_index()
    b = df_b.set_index(ID_COLS)[col].sort_index()
    assert a.equals(b), f"{col} mismatch between sources"

check_target_match(base_train, skill_train)
check_target_match(base_train, form_train)
check_target_match(base_test,  skill_test)
check_target_match(base_test,  form_test)

print("Row-level alignment verified")

Row-level alignment verified


## Step 4: Pre-merge column conflict assertions

In [6]:
# Verify incremental columns are not already present in baseline
# (overlapping names would produce silent _x/_y suffixes in the merge)
for col in SKILL_COLS:
    assert col not in base_train.columns, f"{col} already in base_train — merge will produce _x/_y suffixes"
    assert col not in base_test.columns,  f"{col} already in base_test"
for col in FORM_COLS:
    assert col not in base_train.columns, f"{col} already in base_train"
    assert col not in base_test.columns,  f"{col} already in base_test"

# SKILL and FORM blocks must not overlap (combined dataset requires no clashes)
assert not set(SKILL_COLS) & set(FORM_COLS), "SKILL_COLS and FORM_COLS must not overlap"

# Verify source datasets contain the columns we plan to merge
for col in SKILL_COLS:
    assert col in skill_train.columns, f"Required: {col} not in skill_train"
    assert col in skill_test.columns,  f"Required: {col} not in skill_test"
for col in FORM_COLS:
    assert col in form_train.columns, f"Required: {col} not in form_train"
    assert col in form_test.columns,  f"Required: {col} not in form_test"

print("Pre-merge checks passed")

Pre-merge checks passed


## Step 5: Build skill-aware datasets

In [7]:
skill_aware_train = build_skill_dataset(base_train, skill_train)
skill_aware_test  = build_skill_dataset(base_test,  skill_test)

assert len(skill_aware_train.columns) == n_base_cols + len(SKILL_COLS)
assert len(skill_aware_test.columns)  == n_base_cols + len(SKILL_COLS)
assert skill_aware_train[BASE_COLS].equals(base_train[BASE_COLS])
assert skill_aware_test[BASE_COLS].equals(base_test[BASE_COLS])

print(f"skill_aware_train: {skill_aware_train.shape}")
print(f"skill_aware_test:  {skill_aware_test.shape}")
print("Skill-aware datasets built")

skill_aware_train: (34159, 41)
skill_aware_test:  (8881, 41)
Skill-aware datasets built


## Step 6: Build form-aware datasets

Merge only `FORM_COLS` from the form parquet onto the baseline dataset.
The form parquet also contains skill columns, but we intentionally exclude them here
to create a fair form-only comparison dataset (36 + 8 = 44 cols).

In [8]:
form_aware_train = build_form_dataset(base_train, form_train)
form_aware_test  = build_form_dataset(base_test,  form_test)

assert len(form_aware_train.columns) == n_base_cols + len(FORM_COLS)
assert len(form_aware_test.columns)  == n_base_cols + len(FORM_COLS)
assert form_aware_train[BASE_COLS].equals(base_train[BASE_COLS])
assert form_aware_test[BASE_COLS].equals(base_test[BASE_COLS])

print(f"form_aware_train: {form_aware_train.shape}")
print(f"form_aware_test:  {form_aware_test.shape}")
print("Form-aware datasets built")

form_aware_train: (34159, 44)
form_aware_test:  (8881, 44)
Form-aware datasets built


## Step 7: Build combined datasets

Merge form columns onto the already-built skill-aware datasets — not onto the raw form
parquet, which already contains skill columns and would produce duplicates.

In [9]:
combined_train = build_combined_dataset(skill_aware_train, form_train)
combined_test  = build_combined_dataset(skill_aware_test,  form_test)

assert len(combined_train.columns) == n_base_cols + len(SKILL_COLS) + len(FORM_COLS)
assert len(combined_test.columns)  == n_base_cols + len(SKILL_COLS) + len(FORM_COLS)
assert combined_train[BASE_COLS].equals(base_train[BASE_COLS])
assert combined_test[BASE_COLS].equals(base_test[BASE_COLS])

print(f"combined_train: {combined_train.shape}")
print(f"combined_test:  {combined_test.shape}")
print("Combined datasets built")

combined_train: (34159, 49)
combined_test:  (8881, 49)
Combined datasets built


## Step 8: Missing-value policy

Null after merge = pipeline bug. Do not impute.
Notebooks 05/06 already define valid zero-history defaults;
null here means a shot row was missing from a source file.

In [10]:
# Print null counts per dataset (train only, for readability)
# Null assertions for both splits already ran in Steps 5-7
for name, df, cols in [
    ("skill_aware_train", skill_aware_train, SKILL_COLS),
    ("form_aware_train",  form_aware_train,  FORM_COLS),
    ("combined_train",    combined_train,    SKILL_COLS + FORM_COLS),
]:
    nulls = df[cols].isna().sum()
    print(f"{name}:")
    print(nulls[nulls > 0] if nulls.any() else "  No nulls")
    print()

skill_aware_train:
  No nulls

form_aware_train:
  No nulls

combined_train:
  No nulls



## Step 9: Fairness and consistency across datasets

Verify all three train datasets share identical baseline rows, and same for test.
This is the core fairness guarantee: only the incremental feature blocks differ.

In [11]:
BASE_COMPARE_COLS = ID_COLS + ["is_goal", "xg_pred"]
# BASE_COMPARE_COLS rather than all BASE_COLS — the incremental feature blocks are
# intentionally different across the three datasets; only the shared baseline is compared.

# Row order is preserved (all built from base_train via left merge) so .equals() is valid
assert skill_aware_train[BASE_COMPARE_COLS].equals(form_aware_train[BASE_COMPARE_COLS])
assert skill_aware_train[BASE_COMPARE_COLS].equals(combined_train[BASE_COMPARE_COLS])
assert skill_aware_test[BASE_COMPARE_COLS].equals(form_aware_test[BASE_COMPARE_COLS])
assert skill_aware_test[BASE_COMPARE_COLS].equals(combined_test[BASE_COMPARE_COLS])

print("Fairness checks passed")

Fairness checks passed


## Step 10: Summaries

In [12]:
def summarise(name, train_df, test_df):
    rows = []
    for split, df in [("train", train_df), ("test", test_df)]:
        row = {"dataset": name, "split": split, "rows": len(df),
               "goal_rate": round(df["is_goal"].mean(), 4)}
        if "career_shots_before_shot" in df.columns:
            row["mean_career_shots"] = round(df["career_shots_before_shot"].mean(), 2)
            row["share_with_skill_history"] = round((df["has_prior_shot_history"] > 0).mean(), 4)
        if "shots_last_5_matches" in df.columns:
            row["mean_shots_last_5"] = round(df["shots_last_5_matches"].mean(), 2)
            row["share_with_form_history"] = round((df["matches_in_form_window"] > 0).mean(), 4)
        rows.append(row)
    return pd.DataFrame(rows)

summary = pd.concat([
    summarise("skill_aware", skill_aware_train, skill_aware_test),
    summarise("form_aware",  form_aware_train,  form_aware_test),
    summarise("combined",    combined_train,    combined_test),
], ignore_index=True)
display(summary)

,dataset,split,rows,goal_rate,mean_career_shots,share_with_skill_history,mean_shots_last_5,share_with_form_history
0,skill_aware,train,34159,0.1113,21.75,0.9502,NaN,NaN
1,skill_aware,test,8881,0.1112,21.86,0.9530,NaN,NaN
2,form_aware,train,34159,0.1113,NaN,NaN,8.35,0.9227
3,form_aware,test,8881,0.1112,NaN,NaN,8.43,0.9228
4,combined,train,34159,0.1113,21.75,0.9502,8.35,0.9227
5,combined,test,8881,0.1112,21.86,0.9530,8.43,0.9228


## Step 11: Save outputs

In [13]:
save_dataset_bundle(skill_aware_train, skill_aware_test, "skill_aware", DATA_DIR)
save_dataset_bundle(form_aware_train,  form_aware_test,  "form_aware",  DATA_DIR)
save_dataset_bundle(combined_train,    combined_test,    "combined",    DATA_DIR)

# Optional combined outputs (not part of formal verification contract)
# Notebook 08 consumes the six split outputs above; these are for inspection only.
for train_df, test_df, path in [
    (skill_aware_train, skill_aware_test, DATA_DIR / "wyscout_xg_skill_aware_combined.parquet"),
    (form_aware_train,  form_aware_test,  DATA_DIR / "wyscout_xg_form_aware_combined.parquet"),
    (combined_train,    combined_test,    DATA_DIR / "wyscout_xg_combined_full.parquet"),
]:
    comb = pd.concat([train_df, test_df], ignore_index=True)
    cols = [c for c in comb.columns if c != "dataset_split"]
    comb[cols].to_parquet(path, index=False)

print("All outputs saved")

All outputs saved


## Step 12: Reload verification

In [ ]:
from src.config import TRAIN_LEAGUES, TEST_LEAGUES
from src.datasets import verify_saved_xg_split

skill_aware_train_check = verify_saved_xg_split(
    DATA_DIR / "wyscout_train_xg_skill_aware.parquet",
    expected_rows=n_base_train, required_cols=SKILL_COLS, absent_cols=FORM_COLS,
    expected_leagues=TRAIN_LEAGUES, name="skill_aware_train")
skill_aware_test_check = verify_saved_xg_split(
    DATA_DIR / "wyscout_test_xg_skill_aware.parquet",
    expected_rows=n_base_test, required_cols=SKILL_COLS, absent_cols=FORM_COLS,
    expected_leagues=TEST_LEAGUES, name="skill_aware_test")

form_aware_train_check = verify_saved_xg_split(
    DATA_DIR / "wyscout_train_xg_form_aware.parquet",
    expected_rows=n_base_train, required_cols=FORM_COLS, absent_cols=SKILL_COLS,
    expected_leagues=TRAIN_LEAGUES, name="form_aware_train")
form_aware_test_check = verify_saved_xg_split(
    DATA_DIR / "wyscout_test_xg_form_aware.parquet",
    expected_rows=n_base_test, required_cols=FORM_COLS, absent_cols=SKILL_COLS,
    expected_leagues=TEST_LEAGUES, name="form_aware_test")

combined_train_check = verify_saved_xg_split(
    DATA_DIR / "wyscout_train_xg_combined.parquet",
    expected_rows=n_base_train, required_cols=SKILL_COLS + FORM_COLS,
    expected_leagues=TRAIN_LEAGUES, name="combined_train")
combined_test_check = verify_saved_xg_split(
    DATA_DIR / "wyscout_test_xg_combined.parquet",
    expected_rows=n_base_test, required_cols=SKILL_COLS + FORM_COLS,
    expected_leagues=TEST_LEAGUES, name="combined_test")

print("All reload checks passed")

# ── Notebook-specific: column count verification ──
for name, df in [
    ("skill_aware_train", skill_aware_train_check),
    ("skill_aware_test",  skill_aware_test_check),
]:
    assert len(df.columns) == n_base_cols + len(SKILL_COLS), f"Wrong column count in {name}"
for name, df in [
    ("form_aware_train", form_aware_train_check),
    ("form_aware_test",  form_aware_test_check),
]:
    assert len(df.columns) == n_base_cols + len(FORM_COLS), f"Wrong column count in {name}"
for name, df in [
    ("combined_train", combined_train_check),
    ("combined_test",  combined_test_check),
]:
    assert len(df.columns) == n_base_cols + len(SKILL_COLS) + len(FORM_COLS), \
        f"Wrong column count in {name}"

print("Column count checks passed")
print(f"skill-aware: {skill_aware_train_check.shape[1]} cols  "
      f"| form-aware: {form_aware_train_check.shape[1]} cols  "
      f"| combined: {combined_train_check.shape[1]} cols")

## What notebook 08 will consume

Notebook 08 trains and evaluates four models:

| Model | Input dataset |
|-------|---------------|
| Baseline | `../data/wyscout_train_xg_baseline_predictions.parquet` (notebook 04 output) |
| Skill-aware | `../data/wyscout_train_xg_skill_aware.parquet` (notebook 07 output) |
| Form-aware | `../data/wyscout_train_xg_form_aware.parquet` (notebook 07 output) |
| Combined | `../data/wyscout_train_xg_combined.parquet` (notebook 07 output) |